In [5]:
import re
from typing import Dict, Any, List, Tuple, Optional
from collections import defaultdict

# ===========================
# CONFIGURATION & CONSTANTS
# ===========================

# Competition thresholds
DELTA = 0.3
HIGH_CONF = 0.1
EYE_FALLBACK_THRESHOLD = 0.25

# Limits for multi-value attributes
MAX_HAIR_STYLES = 2
MAX_ACCESSORIES = 5
MAX_CLOTHING_ITEMS = 5
MAX_EXPRESSIONS = 3

# Character counting
CHAR_COUNT_REGEX = re.compile(r"(\d+)\s*(girl|girls|boy|boys)", re.I)
MULTI_KEYWORDS = {"multiple", "several", "group", "couple", "team"}

# ===========================
# BODY MEASUREMENTS TAGS
# ===========================

# Breast size tags (from Danbooru)
BREAST_SIZE_TAGS = {
    "flat_chest": {"size": "flat", "body_type_weight": {"petite": 0.8, "slim": 0.6}},
    "small_breasts": {"size": "small", "body_type_weight": {"slim": 0.5, "petite": 0.4}},
    "medium_breasts": {"size": "medium", "body_type_weight": {"slim": 0.3, "average": 0.5}},
    "large_breasts": {"size": "large", "body_type_weight": {"curvy": 0.6, "voluptuous": 0.4}},
    "huge_breasts": {"size": "huge", "body_type_weight": {"voluptuous": 0.8, "curvy": 0.5}},
    "gigantic_breasts": {"size": "gigantic", "body_type_weight": {"voluptuous": 0.9}},
}

# Additional breast-related tags
BREAST_DESCRIPTOR_TAGS = {
    "sagging_breasts", "perky_breasts", "bouncing_breasts",
    "hanging_breasts", "asymmetrical_breasts", "oppai_loli"
}

# Hip/Butt size tags
HIP_SIZE_TAGS = {
    "narrow_hips": {"size": "narrow", "body_type_weight": {"slim": 0.6, "petite": 0.5}},
    "wide_hips": {"size": "wide", "body_type_weight": {"curvy": 0.7, "voluptuous": 0.5}},
    "huge_hips": {"size": "huge", "body_type_weight": {"voluptuous": 0.8, "thick": 0.6}},
    "big_ass": {"size": "large", "body_type_weight": {"curvy": 0.6, "thick": 0.5}},
    "huge_ass": {"size": "huge", "body_type_weight": {"voluptuous": 0.7, "thick": 0.6}},
}

# Body proportion tags
BODY_PROPORTION_TAGS = {
    "thick_thighs": {"body_type_weight": {"curvy": 0.5, "thick": 0.6, "athletic": 0.3}},
    "muscular_female": {"body_type_weight": {"muscular": 0.8, "athletic": 0.7}},
    "toned": {"body_type_weight": {"athletic": 0.7, "fit": 0.6}},
    "chubby": {"body_type_weight": {"chubby": 0.8, "plump": 0.6}},
    "skinny": {"body_type_weight": {"slim": 0.7, "skinny": 0.8, "petite": 0.5}},
    "plump": {"body_type_weight": {"plump": 0.7, "chubby": 0.5}},
}

# Combined body type tags (original + inferred)
BODY_TYPE_TAGS = {
    "slim", "slender", "petite", "skinny", "thin",
    "curvy", "voluptuous", "hourglass_figure",
    "muscular", "toned", "athletic", "fit",
    "chubby", "plump", "thick", "heavyset",
    "tall", "short", "average_height"
}

# ===========================
# AGE INFERENCE TAGS
# ===========================

# Age mapping (enhanced)
AGE_TAG_MAP = {
    # Child / pre-adolescent
    "loli": "child",
    "shota": "child",
    "child": "child",
    "preteen": "child",
    "toddler": "child",
    "baby": "child",
    "infant": "child",

    # Teen / adolescent
    "teen": "teen",
    "teenager": "teen",
    "high_schooler": "teen",
    "middle_schooler": "teen",

    # Young adult (late teens into 20s/30s)
    "young_adult": "young adult",
    "college_student": "young adult",
    "adult": "young adult",
    "university_student": "young adult",

    # Middle-aged (established adult)
    "middle_aged": "middle-aged",
    "adult_male": "middle-aged",
    "adult_female": "middle-aged",
    "working_adult": "middle-aged",
    "professional": "middle-aged",
    "parent": "middle-aged",

    # Elderly / seniors
    "elderly": "elderly",
    "old_man": "elderly",
    "old_woman": "elderly",
    "grandfather": "elderly",
    "grandmother": "elderly",
    "aged_up": "elderly",
}

# Age inference signals (multi-modal)
AGE_INFERENCE_SIGNALS = {
    # Clothing-based age hints
    "school_uniform": {"age": "teen", "confidence": 0.7},
    "sailor_uniform": {"age": "teen", "confidence": 0.65},
    "serafuku": {"age": "teen", "confidence": 0.65},
    "gakuran": {"age": "teen", "confidence": 0.6},
    "college_uniform": {"age": "young adult", "confidence": 0.6},
    "business_suit": {"age": "middle-aged", "confidence": 0.5},
    "suit": {"age": "young adult", "confidence": 0.4},
    "wedding_dress": {"age": "young adult", "confidence": 0.5},
    "santa_costume": {"age": "young adult", "confidence": 0.3},

    # Height/proportion hints
    "tall": {"age": "young adult", "confidence": 0.3},
    "short": {"age": "teen", "confidence": 0.25},
    "petite": {"age": "teen", "confidence": 0.3},

    # Facial maturity hints
    "mature": {"age": "middle-aged", "confidence": 0.6},
    "mature_female": {"age": "middle-aged", "confidence": 0.65},
    "milf": {"age": "middle-aged", "confidence": 0.7},
    "mature_male": {"age": "middle-aged", "confidence": 0.65},

    # Body development hints (for anime context)
    "flat_chest": {"age": "teen", "confidence": 0.4},
    "small_breasts": {"age": "teen", "confidence": 0.3},
    "oppai_loli": {"age": "child", "confidence": 0.8},
}

# ===========================
# CLOTHING TAGS (SPECIFIC ITEMS)
# ===========================

# Comprehensive clothing item tags (not categories)
SPECIFIC_CLOTHING_TAGS = {
    # Tops
    "shirt", "t-shirt", "polo_shirt", "dress_shirt", "button-up_shirt",
    "blouse", "tank_top", "crop_top", "tube_top", "halter_top",
    "sweater", "cardigan", "hoodie", "pullover", "turtleneck",
    "vest", "waistcoat", "blazer", "jacket", "coat",
    "kimono", "yukata", "hakama", "qipao", "cheongsam",
    "bikini_top", "bra", "sports_bra", "camisole", "bustier",

    # Bottoms
    "pants", "jeans", "trousers", "slacks", "chinos",
    "shorts", "short_shorts", "denim_shorts", "gym_shorts",
    "skirt", "mini_skirt", "pleated_skirt", "pencil_skirt",
    "hakama_skirt", "kilt",

    # Full body
    "dress", "sundress", "cocktail_dress", "evening_gown",
    "wedding_dress", "ballgown", "mini_dress", "maxi_dress",
    "school_uniform", "sailor_uniform", "serafuku", "gakuran",
    "suit", "tuxedo", "business_suit",
    "swimsuit", "one-piece_swimsuit", "bikini",
    "leotard", "bodysuit", "jumpsuit",
    "maid_outfit", "maid_dress", "apron_dress",
    "pajamas", "nightgown", "negligee", "bathrobe",

    # Outerwear
    "trench_coat", "overcoat", "winter_coat", "fur_coat",
    "parka", "windbreaker", "raincoat", "poncho",
    "cape", "cloak", "shawl",

    # Traditional/Cultural
    "hanbok", "sari", "dirndl", "toga", "kesa",

    # Footwear
    "boots", "high_heels", "sandals", "sneakers",
    "loafers", "mary_janes", "platform_shoes",
    "stockings", "thighhighs", "pantyhose", "socks",
    "leg_warmers", "garter_belt",

    # Accessories (wearable)
    "gloves", "mittens", "arm_warmers", "fingerless_gloves",
    "scarf", "necktie", "bowtie", "ribbon", "ascot",
    "belt", "sash", "suspenders",
    "apron", "armband", "wristband",
}

# Eye colors
EYE_COLORS = {
    "blue_eyes", "brown_eyes", "green_eyes", "red_eyes",
    "purple_eyes", "yellow_eyes", "pink_eyes", "orange_eyes",
    "grey_eyes", "heterochromia", "multicolored_eyes"
}

# Hair colors
HAIR_COLOR_TAGS = {
    "black_hair", "brown_hair", "blonde_hair", "white_hair",
    "silver_hair", "grey_hair", "red_hair", "pink_hair",
    "purple_hair", "blue_hair", "green_hair", "orange_hair",
    "multicolored_hair", "gradient_hair", "two-tone_hair"
}

# Hair styles
HAIR_STYLE_TAGS = {
    "ponytail", "twintails", "braid", "twin_braids", "side_braid",
    "bun", "twin_buns", "side_bun", "bob_cut", "hime_cut",
    "straight_hair", "wavy_hair", "curly_hair", "drill_hair",
    "ahoge", "hair_bun", "messy_hair", "spiked_hair",
    "slicked_back_hair", "pixie_cut", "undercut"
}

# Accessories
ACCESSORY_TAGS = {
    # Head / face accessories
    "glasses", "sunglasses", "eyewear", "visor", "goggles",
    "hat", "beret", "cap", "headband", "hairband",
    "hair_bow", "hairclip", "hair_ornament", "ribbon",
    "tiara", "crown", "bandana",

    # Jewelry
    "earrings", "hoop_earrings", "stud_earrings",
    "necklace", "choker", "pendant",
    "bracelet", "anklet", "ring",
    "body_chain", "brooch", "pin",

    # Neck / shoulder
    "scarf", "necktie", "bowtie", "cravat",

    # Bags / carrying
    "backpack", "bag", "handbag", "pouch",
    "satchel", "crossbody_bag", "fanny_pack",

    # Hand / wrist
    "gloves", "fingerless_gloves",
    "armbands", "wristbands", "watch",

    # Leg / foot
    "stockings", "thighhighs", "socks",
    "leg_warmers", "belt", "sash",

    # Misc wearable accessories
    "mask", "face_mask", "headset", "earphones",
    "tie_clip", "cufflinks", "corset", "vest",
    "cape", "shawl", "cloak"
}

EXPRESSION_TAGS = {
    # Positive / happy
    "smile", "grin", "happy", "blush", "smirking",
    "cheerful", "gleeful", "laughing", "content", "ecstatic",

    # Neutral / subtle
    "neutral", "expressionless", "calm", "relaxed",
    "serious", "poker_face", "straight_face", "stoic",

    # Negative / sadness
    "sad", "tears", "crying", "depressed", "upset",
    "frowning", "downcast", "mourning",

    # Anger / annoyance
    "angry", "annoyed", "irritated", "scowl",
    "grumpy", "shouting", "yelling", "fierce", "rage",

    # Surprise / shock
    "surprised", "shocked", "astonished", "wide_eyes", "gasp",

    # Playful / mischievous
    "wink", "teasing", "mischievous", "cheeky",
    "playful", "tongue_out",

    # Affection / love
    "love", "heart_eyes", "affectionate",
    "blowing_kiss", "adoring",

    # Embarrassed / shy
    "embarrassed", "bashful", "flustered", "red_face",

    # Other emotion/pose indicators
    "smug", "determined", "pensive", "serene",
    "intense", "confident"
}

# Clothing categories (kept for backward compatibility)
CLOTHING_MAP = {
    "uniform": {
        "school_uniform", "sailor_uniform", "military_uniform",
        "gakuran", "serafuku", "summer_uniform", "winter_uniform",
    },
    "traditional": {
        "kimono", "yukata", "hakama", "hanbok", "sari",
        "japanese_clothes", "traditional_clothes", "obi",
        "chinese_clothes", "qipao", "cheongsam"
    },
    "formal": {
        "suit", "tuxedo", "formal_dress", "evening_gown",
        "cocktail_dress", "wedding_dress", "business_suit"
    },
    "casual": {
        "hoodie", "jacket", "blazer", "shirt", "tshirt",
        "crop_top", "tank_top", "hooded_sweater", "sweater",
        "jeans", "shorts", "pants", "denim_jacket", "cardigan"
    },
    "swimwear": {
        "swimsuit", "bikini", "swim_briefs", "string_bikini",
        "sports_bikini", "one-piece_swimsuit"
    },
    "dress": {
        "dress", "sundress", "pleated_skirt", "skirt",
        "micro_dress", "ballgown", "halter_dress", "mini_dress"
    },
    "outerwear": {
        "coat", "trench_coat", "parka", "fur_coat",
        "cape", "cloak", "poncho", "raincoat"
    },
    "footwear": {
        "boots", "sandals", "sneakers", "heels",
        "loafers", "barefoot", "socks", "stockings"
    },
    "costume": {
        "maid_outfit", "cheerleader_uniform", "bunny_costume",
        "cat_costume", "dog_costume", "animal_costume",
        "christmas_costume", "halloween_costume", "cosplay"
    },
    "sleepwear": {
        "pajamas", "nightgown", "negligee", "bathrobe"
    },
}

# Anime origin tags
ANIME_ORIGIN_TAGS = {
    "japanese": {"anime", "manga", "light_novel", "visual_novel", "japanese"},
    "korean": {"manhwa", "webtoon", "korean"},
    "chinese": {"manhua", "donghua", "chinese"},
    "western": {"western", "cartoon", "comic"}
}

# ===========================
# MALE-SPECIFIC BODY TYPE INFERENCE
# ===========================

# Male body measurement tags
MALE_BODY_TAGS = {
    # Muscle definition
    "muscular_male": {"body_type_weight": {"muscular": 0.9, "athletic": 0.6}},
    "muscular": {"body_type_weight": {"muscular": 0.8, "athletic": 0.5}},
    "toned": {"body_type_weight": {"athletic": 0.8, "fit": 0.7}},
    "bara": {"body_type_weight": {"muscular": 0.9, "bulky": 0.7}},

    # Chest/torso development
    "pectorals": {"body_type_weight": {"muscular": 0.6, "athletic": 0.5}},
    "large_pectorals": {"body_type_weight": {"muscular": 0.8, "bulky": 0.6}},
    "abs": {"body_type_weight": {"athletic": 0.7, "fit": 0.6}},
    "six-pack": {"body_type_weight": {"athletic": 0.8, "muscular": 0.6}},

    # Build/frame
    "broad_shoulders": {"body_type_weight": {"muscular": 0.6, "athletic": 0.5}},
    "thick_arms": {"body_type_weight": {"muscular": 0.7, "athletic": 0.4}},
    "thick_neck": {"body_type_weight": {"muscular": 0.6, "bulky": 0.5}},
    "strongman_waist": {"body_type_weight": {"bulky": 0.8, "muscular": 0.5}},

    # Slim/lean builds
    "skinny_male": {"body_type_weight": {"slim": 0.8, "skinny": 0.9}},
    "thin": {"body_type_weight": {"slim": 0.7, "skinny": 0.6}},
    "slender": {"body_type_weight": {"slim": 0.6, "lean": 0.5}},
    "lean": {"body_type_weight": {"athletic": 0.5, "slim": 0.4}},

    # Heavy/larger builds
    "chubby_male": {"body_type_weight": {"chubby": 0.8, "heavyset": 0.5}},
    "fat_man": {"body_type_weight": {"heavyset": 0.9, "obese": 0.7}},
    "belly": {"body_type_weight": {"heavyset": 0.6, "chubby": 0.5}},
    "dad_bod": {"body_type_weight": {"average": 0.6, "chubby": 0.4}},
    "beer_belly": {"body_type_weight": {"chubby": 0.7, "heavyset": 0.5}},

    # Height/stature
    "tall_male": {"body_type_weight": {"tall": 0.7}},
    "short_male": {"body_type_weight": {"short": 0.7}},
    "giant": {"body_type_weight": {"tall": 0.9, "bulky": 0.5}},

    # Special builds
    "otoko_no_ko": {"body_type_weight": {"slim": 0.7, "petite": 0.5}},
    "femboy": {"body_type_weight": {"slim": 0.8, "petite": 0.6}},
    "trap_(gender)": {"body_type_weight": {"slim": 0.7, "petite": 0.5}},
}

# Male body hair (secondary characteristic for body type + age)
MALE_BODY_HAIR_TAGS = {
    "chest_hair": {"body_type_weight": {"muscular": 0.3, "mature": 0.4}, "age_hint": "middle-aged"},
    "arm_hair": {"body_type_weight": {"muscular": 0.2, "mature": 0.3}, "age_hint": "young adult"},
    "leg_hair": {"body_type_weight": {"athletic": 0.2}, "age_hint": "young adult"},
    "hairy": {"body_type_weight": {"mature": 0.5}, "age_hint": "middle-aged"},
    "hairy_male": {"body_type_weight": {"mature": 0.6}, "age_hint": "middle-aged"},
    "body_hair": {"body_type_weight": {"mature": 0.4}, "age_hint": "young adult"},
}

# ===========================
# MALE-SPECIFIC AGE INFERENCE
# ===========================

# Facial hair as strong age indicator
MALE_FACIAL_HAIR_AGE = {
    "stubble": {"age": "young adult", "confidence": 0.6},
    "5_o'clock_shadow": {"age": "young adult", "confidence": 0.5},
    "beard": {"age": "middle-aged", "confidence": 0.7},
    "full_beard": {"age": "middle-aged", "confidence": 0.75},
    "goatee": {"age": "young adult", "confidence": 0.55},
    "mustache": {"age": "middle-aged", "confidence": 0.65},
    "long_beard": {"age": "elderly", "confidence": 0.8},
    "white_beard": {"age": "elderly", "confidence": 0.85},
    "sideburns": {"age": "young adult", "confidence": 0.4},
    "soul_patch": {"age": "young adult", "confidence": 0.5},
}

# Male-specific facial features for age
MALE_FACIAL_AGE_HINTS = {
    # Youth markers
    "round_face": {"age": "teen", "confidence": 0.5},
    "soft_features": {"age": "teen", "confidence": 0.4},
    "large_eyes": {"age": "teen", "confidence": 0.3},
    "smooth_skin": {"age": "teen", "confidence": 0.4},

    # Young adult markers (20s-30s)
    "sharp_jawline": {"age": "young adult", "confidence": 0.5},
    "defined_jawline": {"age": "young adult", "confidence": 0.5},
    "chiseled_features": {"age": "young adult", "confidence": 0.6},
    "narrow_eyes": {"age": "young adult", "confidence": 0.3},

    # Middle-age markers (40s-50s)
    "wrinkles": {"age": "middle-aged", "confidence": 0.7},
    "eye_wrinkles": {"age": "middle-aged", "confidence": 0.65},
    "forehead_wrinkles": {"age": "middle-aged", "confidence": 0.6},
    "defined_cheekbones": {"age": "middle-aged", "confidence": 0.5},
    "mature_male": {"age": "middle-aged", "confidence": 0.8},
    "dilf": {"age": "middle-aged", "confidence": 0.85},

    # Elderly markers (60+)
    "receding_hairline": {"age": "middle-aged", "confidence": 0.6},
    "bald": {"age": "middle-aged", "confidence": 0.5},
    "bald_male": {"age": "middle-aged", "confidence": 0.55},
    "grey_hair": {"age": "elderly", "confidence": 0.7},
    "white_hair": {"age": "elderly", "confidence": 0.6},
    "sagging_skin": {"age": "elderly", "confidence": 0.8},
    "old_man": {"age": "elderly", "confidence": 0.95},
    "elderly_male": {"age": "elderly", "confidence": 0.9},
    "grandfather": {"age": "elderly", "confidence": 0.85},
}

# Male-specific clothing age hints
MALE_CLOTHING_AGE_HINTS = {
    # Youth/Teen
    "gakuran": {"age": "teen", "confidence": 0.7},
    "school_uniform": {"age": "teen", "confidence": 0.65},
    "gym_uniform": {"age": "teen", "confidence": 0.6},
    "track_suit": {"age": "teen", "confidence": 0.5},
    "hoodie": {"age": "teen", "confidence": 0.35},
    "backpack": {"age": "teen", "confidence": 0.4},

    # Young adult
    "tuxedo": {"age": "young adult", "confidence": 0.6},
    "business_suit": {"age": "young adult", "confidence": 0.55},
    "dress_shirt": {"age": "young adult", "confidence": 0.4},
    "necktie": {"age": "young adult", "confidence": 0.45},
    "casual_suit": {"age": "young adult", "confidence": 0.5},

    # Middle-aged
    "salaryman": {"age": "middle-aged", "confidence": 0.75},
    "businessman": {"age": "middle-aged", "confidence": 0.7},
    "three_piece_suit": {"age": "middle-aged", "confidence": 0.65},
    "suspenders": {"age": "middle-aged", "confidence": 0.5},
    "vest": {"age": "middle-aged", "confidence": 0.4},

    # Elderly
    "traditional_clothes": {"age": "elderly", "confidence": 0.4},
    "kimono": {"age": "middle-aged", "confidence": 0.3},
    "walking_stick": {"age": "elderly", "confidence": 0.8},
    "cane": {"age": "elderly", "confidence": 0.8},
}

# Male body build age correlation
MALE_BUILD_AGE_HINTS = {
    # Skinny/underdeveloped = younger
    "skinny": {"age": "teen", "confidence": 0.3},
    "thin": {"age": "teen", "confidence": 0.25},

    # Athletic/muscular = young adult prime
    "muscular": {"age": "young adult", "confidence": 0.4},
    "athletic": {"age": "young adult", "confidence": 0.45},
    "toned": {"age": "young adult", "confidence": 0.4},

    # Dad bod/belly = middle-aged
    "dad_bod": {"age": "middle-aged", "confidence": 0.7},
    "beer_belly": {"age": "middle-aged", "confidence": 0.65},
    "belly": {"age": "middle-aged", "confidence": 0.4},
}

# ===========================
# MALE-SPECIFIC HAIR STYLES
# ===========================

# Male hair style tags
MALE_HAIR_STYLE_TAGS = {
    # Very short
    "bald": {"style": "bald", "length": "none"},
    "buzz_cut": {"style": "buzz cut", "length": "very_short"},
    "crew_cut": {"style": "crew cut", "length": "very_short"},
    "flat_top": {"style": "flat top", "length": "very_short"},

    # Short styles
    "short_hair": {"style": "short", "length": "short"},
    "undercut": {"style": "undercut", "length": "short"},
    "sidecut": {"style": "sidecut", "length": "short"},
    "fade": {"style": "fade", "length": "short"},
    "military_cut": {"style": "military cut", "length": "very_short"},

    # Medium/styled
    "pompadour": {"style": "pompadour", "length": "medium"},
    "slicked_back_hair": {"style": "slicked back", "length": "medium"},
    "hair_slicked_back": {"style": "slicked back", "length": "medium"},
    "quiff": {"style": "quiff", "length": "medium"},
    "curtained_hair": {"style": "curtains", "length": "medium"},
    "middle_part": {"style": "middle part", "length": "medium"},
    "side_part": {"style": "side part", "length": "short"},

    # Edgy/alternative
    "mohawk": {"style": "mohawk", "length": "short"},
    "spiky_hair": {"style": "spiky", "length": "short"},
    "spiked_hair": {"style": "spiked", "length": "short"},
    "faux_hawk": {"style": "faux hawk", "length": "short"},
    "dreadlocks": {"style": "dreadlocks", "length": "long"},

    # Long styles (less common for males)
    "long_hair": {"style": "long", "length": "long"},
    "ponytail": {"style": "ponytail", "length": "long"},
    "man_bun": {"style": "man bun", "length": "medium"},
    "topknot": {"style": "topknot", "length": "medium"},
    "samurai_topknot": {"style": "samurai topknot", "length": "medium"},

    # Messy/natural
    "messy_hair": {"style": "messy", "length": "varies"},
    "unkempt_hair": {"style": "unkempt", "length": "varies"},
    "shaggy_hair": {"style": "shaggy", "length": "medium"},
    "tousled_hair": {"style": "tousled", "length": "medium"},

    # Facial hair (part of overall hairstyle)
    "sideburns": {"style": "sideburns", "length": "short"},
    "mutton_chops": {"style": "mutton chops", "length": "medium"},
}

# ===========================
# HELPER FUNCTIONS
# ===========================

def estimate_character_count(tags: List[str]) -> Tuple[int, bool]:
    """
    Estimates character count from DeepDanbooru tags.
    Returns: (count, is_ambiguous)
    """
    total_count = 0
    has_solo = False
    has_multi_keyword = False

    for tag in tags:
        tag_clean = tag.lower().strip()

        if tag_clean == "solo":
            has_solo = True

        match = CHAR_COUNT_REGEX.match(tag_clean)
        if match:
            total_count += int(match.group(1))

        if any(keyword in tag_clean for keyword in MULTI_KEYWORDS):
            has_multi_keyword = True

    if has_solo:
        return 1, False

    if total_count == 0:
        return (2, True) if has_multi_keyword else (0, False)

    if total_count == 1:
        return (2, True) if has_multi_keyword else (1, True)

    return (total_count + 2, True) if has_multi_keyword else (total_count, True)


def _resolve_competition(candidates: Dict[str, float]) -> Tuple[str, float]:
    """
    OPTIMIZATION #7: Combined filter + sort in one pass.
    Resolves competition between candidate attributes.
    Returns: (winner, confidence) or ("unknown", 1.0) if no candidates
    """
    if not candidates:
        return "unknown", 1.0

    # Combined filter + sort - avoid creating intermediate dict
    sorted_items = sorted(
        ((k, v) for k, v in candidates.items() if v > 0),
        key=lambda x: x[1],
        reverse=True
    )

    if not sorted_items:
        return "unknown", 1.0

    if len(sorted_items) == 1:
        return sorted_items[0]

    (v1, p1), (v2, p2) = sorted_items[:2]

    if abs(p1 - p2) < DELTA:
        return v1, p1

    return v1, p1


def _extract_attribute(tag_probs: Dict[str, float], tag_set: set,
                       fallback_threshold: Optional[float] = None) -> Tuple[str, float]:
    """
    OPTIMIZATION #2: Generic attribute extraction helper.
    Reduces repeated pattern of candidate extraction + competition resolution.
    """
    candidates = {
        tag.replace("_eyes", "").replace("_hair", ""): prob
        for tag, prob in tag_probs.items()
        if tag in tag_set and prob >= HIGH_CONF
    }

    if not candidates and fallback_threshold:
        candidates = {
            tag.replace("_eyes", "").replace("_hair", ""): prob
            for tag, prob in tag_probs.items()
            if tag in tag_set and prob >= fallback_threshold
        }

    return _resolve_competition(candidates)


def _infer_body_type_from_measurements(
    tag_probs: Dict[str, float]
) -> Tuple[str, float, List[str]]:
    """
    Enhanced body type inference using breast size, hip size, and body proportions.
    Returns: (body_type, confidence, evidence_list)
    """
    body_type_scores = defaultdict(float)
    evidence = []

    # Step 1: Check breast size tags
    for breast_tag, data in BREAST_SIZE_TAGS.items():
        if breast_tag in tag_probs and tag_probs[breast_tag] > HIGH_CONF:
            prob = tag_probs[breast_tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight
            evidence.append(f"{breast_tag}:{prob:.2f}")

    # Step 2: Check hip/butt size tags
    for hip_tag, data in HIP_SIZE_TAGS.items():
        if hip_tag in tag_probs and tag_probs[hip_tag] > HIGH_CONF:
            prob = tag_probs[hip_tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight
            evidence.append(f"{hip_tag}:{prob:.2f}")

    # Step 3: Check body proportion tags
    for prop_tag, data in BODY_PROPORTION_TAGS.items():
        if prop_tag in tag_probs and tag_probs[prop_tag] > HIGH_CONF:
            prob = tag_probs[prop_tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight
            evidence.append(f"{prop_tag}:{prob:.2f}")

    # Step 4: Check direct body type tags
    for body_tag in BODY_TYPE_TAGS:
        if body_tag in tag_probs and tag_probs[body_tag] > HIGH_CONF:
            prob = tag_probs[body_tag]
            body_type_scores[body_tag] += prob * 1.5
            evidence.append(f"{body_tag}:{prob:.2f}(direct)")

    if not body_type_scores:
        return "unknown", 1.0, []

    body_type, conf = _resolve_competition(dict(body_type_scores))

    return body_type, conf, evidence[:5]


def _infer_male_body_type(tag_probs: Dict[str, float]) -> Tuple[str, float, List[str]]:
    """
    Male-specific body type inference using muscle definition, build, and body hair.
    Returns: (body_type, confidence, evidence_list)
    """
    body_type_scores = defaultdict(float)
    evidence = []

    # Step 1: Check male-specific body tags
    for tag, data in MALE_BODY_TAGS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight
            evidence.append(f"{tag}:{prob:.2f}")

    # Step 2: Check body hair (maturity indicator)
    for tag, data in MALE_BODY_HAIR_TAGS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight * 0.5
            evidence.append(f"{tag}:{prob:.2f}(hair)")

    # Step 3: Fallback to general body tags
    for body_tag in BODY_TYPE_TAGS:
        if body_tag in tag_probs and tag_probs[body_tag] > HIGH_CONF:
            prob = tag_probs[body_tag]
            body_type_scores[body_tag] += prob * 1.2
            evidence.append(f"{body_tag}:{prob:.2f}(general)")

    if not body_type_scores:
        return "unknown", 1.0, []

    body_type, conf = _resolve_competition(dict(body_type_scores))

    return body_type, conf, evidence[:5]


def _infer_age_multimodal(
    tag_probs: Dict[str, float],
    gender: str,
    breast_size: Optional[str] = None,
    clothing_items: List[str] = []
) -> Tuple[str, float, List[str]]:
    """
    Multi-modal age inference using:
    - Direct age tags
    - Clothing context
    - Body development
    - Height/proportion hints
    - Facial maturity tags

    Returns: (age, confidence, evidence_list)
    """
    age_scores = defaultdict(float)
    evidence = []

    # Step 1: Direct age tags (highest weight)
    for tag, mapped_age in AGE_TAG_MAP.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[mapped_age] += prob * 2.0
            evidence.append(f"direct:{tag}:{prob:.2f}")

    # Step 2: Clothing-based age hints
    for item in clothing_items:
        if item in AGE_INFERENCE_SIGNALS:
            signal = AGE_INFERENCE_SIGNALS[item]
            age_scores[signal["age"]] += signal["confidence"]
            evidence.append(f"clothing:{item}:{signal['confidence']:.2f}")

    # Step 3: Additional contextual tags
    for tag, signal in AGE_INFERENCE_SIGNALS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"]
            evidence.append(f"context:{tag}:{prob:.2f}")

    # Step 4: Breast size inference (anime-specific, for female characters)
    if gender == "female" and breast_size:
        if breast_size in ["flat", "small"]:
            age_scores["teen"] += 0.4
            evidence.append(f"breast_size:{breast_size}:0.40")
        elif breast_size == "oppai_loli":
            age_scores["child"] += 0.8
            evidence.append(f"breast_size:oppai_loli:0.80")
        elif breast_size in ["huge", "gigantic"]:
            age_scores["young adult"] += 0.3
            age_scores["middle-aged"] += 0.2
            evidence.append(f"breast_size:{breast_size}:0.30")

    # Step 5: Height/stature inference
    if "tall" in tag_probs and tag_probs["tall"] > HIGH_CONF:
        age_scores["young adult"] += 0.3
        evidence.append(f"height:tall:{tag_probs['tall']:.2f}")
    elif "petite" in tag_probs and tag_probs["petite"] > HIGH_CONF:
        age_scores["teen"] += 0.3
        evidence.append(f"height:petite:{tag_probs['petite']:.2f}")

    if not age_scores:
        return "unknown", 1.0, []

    age, conf = _resolve_competition(dict(age_scores))

    max_possible_score = 3.0
    normalized_conf = min(conf / max_possible_score, 1.0)

    return age, normalized_conf, evidence[:7]


def _infer_male_age(
    tag_probs: Dict[str, float],
    body_type: str,
    clothing_items: List[str] = []
) -> Tuple[str, float, List[str]]:
    """
    Male-specific age inference using:
    - Direct age tags
    - Facial hair (STRONG signal for males)
    - Facial features
    - Body build maturity
    - Clothing context

    Returns: (age, confidence, evidence_list)
    """
    age_scores = defaultdict(float)
    evidence = []

    # Step 1: Direct age tags
    for tag, mapped_age in AGE_TAG_MAP.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[mapped_age] += prob * 2.0
            evidence.append(f"direct:{tag}:{prob:.2f}")

    # Step 2: Facial hair (VERY strong signal)
    for tag, signal in MALE_FACIAL_HAIR_AGE.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"] * 1.5
            evidence.append(f"facial_hair:{tag}:{prob:.2f}")

    # Step 3: Facial features
    for tag, signal in MALE_FACIAL_AGE_HINTS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"]
            evidence.append(f"facial:{tag}:{prob:.2f}")

    # Step 4: Clothing-based age hints
    for item in clothing_items:
        item_tag = item.replace(" ", "_")
        if item_tag in MALE_CLOTHING_AGE_HINTS:
            signal = MALE_CLOTHING_AGE_HINTS[item_tag]
            age_scores[signal["age"]] += signal["confidence"]
            evidence.append(f"clothing:{item}:{signal['confidence']:.2f}")

    # Step 5: Additional clothing tags
    for tag, signal in MALE_CLOTHING_AGE_HINTS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"]
            evidence.append(f"clothing:{tag}:{prob:.2f}")

    # Step 6: Body build age correlation
    for tag, signal in MALE_BUILD_AGE_HINTS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"]
            evidence.append(f"build:{tag}:{prob:.2f}")

    # Step 7: Body hair as age hint
    for tag, data in MALE_BODY_HAIR_TAGS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            if "age_hint" in data:
                age_scores[data["age_hint"]] += prob * 0.4
                evidence.append(f"body_hair:{tag}:{prob:.2f}")

    if not age_scores:
        return "unknown", 1.0, []

    age, conf = _resolve_competition(dict(age_scores))

    max_possible_score = 4.0
    normalized_conf = min(conf / max_possible_score, 1.0)

    return age, normalized_conf, evidence[:7]


def _extract_male_hair_styles(tag_probs: Dict[str, float]) -> List[Dict[str, Any]]:
    """
    Extract male-specific hair styles with length information.
    Returns list of {value, confidence, length} dictionaries.
    """
    hair_styles = []

    for tag, data in MALE_HAIR_STYLE_TAGS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            hair_styles.append({
                "value": data["style"],
                "confidence": round(tag_probs[tag], 3),
                "length": data["length"]
            })

    hair_styles.sort(key=lambda x: x["confidence"], reverse=True)

    return hair_styles[:MAX_HAIR_STYLES]


def infer_body_type_gender_aware(
    tag_probs: Dict[str, float],
    gender: str
) -> Tuple[str, float, List[str]]:
    """
    HOT-SWAPPABLE: Routes to gender-specific body type inference.
    """
    if gender == "male":
        return _infer_male_body_type(tag_probs)
    else:
        return _infer_body_type_from_measurements(tag_probs)


def infer_age_gender_aware(
    tag_probs: Dict[str, float],
    gender: str,
    body_type: str,
    breast_size: Optional[str] = None,
    clothing_items: List[str] = []
) -> Tuple[str, float, List[str]]:
    """
    HOT-SWAPPABLE: Routes to gender-specific age inference.
    """
    if gender == "male":
        return _infer_male_age(tag_probs, body_type, clothing_items)
    else:
        return _infer_age_multimodal(tag_probs, gender, breast_size, clothing_items)


def extract_hair_styles_gender_aware(
    tag_probs: Dict[str, float],
    gender: str
) -> List[Dict[str, Any]]:
    """
    HOT-SWAPPABLE: Routes to gender-specific hair style extraction.
    """
    if gender == "male":
        return _extract_male_hair_styles(tag_probs)
    else:
        hair_style_tags = {t for t in tag_probs if t in HAIR_STYLE_TAGS}
        return _select_top_n(tag_probs, hair_style_tags, MAX_HAIR_STYLES)


def _extract_specific_clothing(tag_probs: Dict[str, float]) -> List[Dict[str, Any]]:
    """
    Extract specific clothing items (not categories) from tags.
    Returns list of {item, confidence} dictionaries.
    """
    clothing_items = []

    for tag, prob in tag_probs.items():
        if tag in SPECIFIC_CLOTHING_TAGS and prob > HIGH_CONF:
            clothing_items.append({
                "item": tag.replace("_", " "),
                "confidence": round(prob, 3)
            })

    clothing_items.sort(key=lambda x: x["confidence"], reverse=True)

    return clothing_items[:MAX_CLOTHING_ITEMS]


def _select_top_n(tag_probs: Dict[str, float], valid_tags: set, n: int) -> List[Dict[str, float]]:
    """Select top N tags from valid set, with confidence scores"""
    filtered = [(t, p) for t, p in tag_probs.items() if t in valid_tags and p > 0]
    filtered.sort(key=lambda x: x[1], reverse=True)
    return [{"value": t, "confidence": p} for t, p in filtered[:n]]


def _infer_skin_tone_from_context(
    tag_probs: Dict[str, float],
    hair_color: str,
    eye_color: str
) -> Tuple[str, float]:
    """
    Infer skin tone from hair and eye color when direct tags are missing.
    Uses common anime art style associations.
    """
    inference_map = {
        ("blonde", "blue"): ("light", 0.4),
        ("blonde", "green"): ("light", 0.4),
        ("silver", "blue"): ("very_light", 0.4),
        ("white", "red"): ("very_light", 0.4),
        ("black", "brown"): ("medium", 0.35),
        ("black", "black"): ("medium", 0.35),
        ("brown", "brown"): ("light", 0.35),
        ("red", "green"): ("light", 0.3),
    }

    key = (hair_color, eye_color)
    if key in inference_map:
        return inference_map[key]

    if hair_color == "black":
        return "medium", 0.25
    elif hair_color in ["blonde", "silver", "white"]:
        return "light", 0.25

    return "unknown", 1.0


def _determine_anime_origin(tag_probs: Dict[str, float]) -> Tuple[str, float]:
    """Determine the origin/style of the anime content"""
    origin_scores = {}

    for origin, tags in ANIME_ORIGIN_TAGS.items():
        # OPTIMIZATION #4: Generator expression instead of list comprehension
        score = max((tag_probs.get(t, 0.0) for t in tags), default=0.0)
        origin_scores[origin] = score

    if not any(score > 0 for score in origin_scores.values()):
        return "japanese", 0.7

    origin, conf = max(origin_scores.items(), key=lambda x: x[1])
    return origin, conf


def _generate_scene_description(tag_probs: Dict[str, float]) -> Dict[str, Any]:
    """Generate description for images without characters"""
    sorted_tags = sorted(tag_probs.items(), key=lambda x: x[1], reverse=True)[:10]

    scene_type = "unknown"
    if any(t in tag_probs for t in ["landscape", "scenery", "outdoors", "nature"]):
        scene_type = "landscape"
    elif any(t in tag_probs for t in ["vehicle", "car", "mecha", "robot"]):
        scene_type = "vehicle"
    elif any(t in tag_probs for t in ["building", "architecture", "city", "urban"]):
        scene_type = "architecture"
    elif any(t in tag_probs for t in ["object", "still_life", "food"]):
        scene_type = "object"
    elif any(t in tag_probs for t in ["abstract", "pattern", "texture"]):
        scene_type = "abstract"

    return {
        "image_status": "no_character_detected",
        "scene_type": scene_type,
        "top_tags": [{"tag": t, "confidence": round(c, 3)} for t, c in sorted_tags]
    }


# ===========================
# MAIN PROJECTOR
# ===========================

def run_projector(tag_probs: Dict[str, float]) -> Dict[str, Any]:
    """
    OPTIMIZATION #1: Single unified run_projector() implementation.
    Enhanced gender-aware projector using inference modules.
    OPTIMIZATION #6: Removed cultural_elements and ethnicity_estimate.
    OPTIMIZATION #8: Pre-filter high confidence tags.
    """
    tag_list = list(tag_probs.keys())

    # OPTIMIZATION #8: Pre-filter high confidence tags once
    high_conf_tags = {tag: prob for tag, prob in tag_probs.items() if prob > HIGH_CONF}

    # Character count gate
    count, ambiguous = estimate_character_count(tag_list)

    if count == 0:
        return _generate_scene_description(tag_probs)

    if count != 1:
        return {
            "image_status": "ambiguous_multi_character",
            "character_count": count,
            "attributes": None
        }

    output = {"image_status": "single_character"}

    # ---- Gender (FIRST - needed for routing) ----
    gender, g_conf = _extract_attribute(tag_probs, {"1girl", "1boy"})
    gender = "female" if gender == "1girl" else "male" if gender == "1boy" else gender
    output["gender"] = gender
    output["gender_confidence"] = round(g_conf, 3)

    # ---- Eye Color ----
    eye_color, e_conf = _extract_attribute(tag_probs, EYE_COLORS, EYE_FALLBACK_THRESHOLD)
    output["eye_color"] = eye_color
    output["eye_color_confidence"] = round(e_conf, 3)

    # ---- Hair Color ----
    hair_color, hc_conf = _extract_attribute(tag_probs, HAIR_COLOR_TAGS)
    output["hair_color"] = hair_color
    output["hair_color_confidence"] = round(hc_conf, 3)

    # ---- Hair Length ----
    hair_length_tags = {
        "long_hair", "short_hair", "medium_hair", "shoulder_length_hair",
        "midback_length_hair", "waist_length_hair", "hip_length_hair", "ankle_length_hair"
    }
    hair_length_candidates = {}
    for tag in hair_length_tags:
        if tag in tag_probs:
            clean_name = tag.replace("_hair", "").replace("_length", "")
            hair_length_candidates[clean_name] = tag_probs[tag]

    hair_length, hl_conf = _resolve_competition(hair_length_candidates)
    output["hair_length"] = hair_length
    output["hair_length_confidence"] = round(hl_conf, 3)

    # ---- Hair Styles (GENDER-AWARE) ----
    hair_styles = extract_hair_styles_gender_aware(tag_probs, gender)
    output["hair_styles"] = hair_styles

    # ---- Skin Tone ----
    skin_tags = {
        "pale_skin", "light_skin", "tan_skin", "medium_skin", "olive_skin",
        "brown_skin", "dark_skin", "very_dark_skin", "colored_skin"
    }
    skin_candidates = {}
    for tag in skin_tags:
        if tag in tag_probs:
            clean_name = tag.replace("_skin", "")
            if clean_name == "pale":
                clean_name = "very_light"
            elif clean_name == "colored":
                clean_name = "colored"
            skin_candidates[clean_name] = tag_probs[tag]

    skin, s_conf = _resolve_competition(skin_candidates)
    if skin == "unknown":
        skin, s_conf = _infer_skin_tone_from_context(tag_probs, hair_color, eye_color)
    output["skin_tone"] = skin
    output["skin_tone_confidence"] = round(s_conf, 3)

    # ---- Body Type (GENDER-AWARE) ----
    body_type, bt_conf, body_evidence = infer_body_type_gender_aware(tag_probs, gender)
    output["body_type"] = body_type
    output["body_type_confidence"] = round(bt_conf, 3)
    output["body_type_evidence"] = body_evidence

    # ---- Extract breast size (for female age inference) ----
    breast_size = None
    if gender == "female":
        for breast_tag, data in BREAST_SIZE_TAGS.items():
            if breast_tag in high_conf_tags:
                breast_size = data["size"]
                break

    # ---- Clothing Items ----
    specific_clothing = _extract_specific_clothing(tag_probs)
    output["clothing_items"] = specific_clothing

    clothing_categories = []
    for category, tags in CLOTHING_MAP.items():
        # OPTIMIZATION #4: Generator expression instead of list
        max_score = max((tag_probs.get(t, 0.0) for t in tags), default=0.0)
        if max_score > HIGH_CONF:
            clothing_categories.append({"category": category, "confidence": round(max_score, 3)})
    clothing_categories.sort(key=lambda x: x["confidence"], reverse=True)
    output["clothing"] = clothing_categories[:MAX_CLOTHING_ITEMS]

    # ---- Age (GENDER-AWARE) ----
    clothing_item_names = [item["item"].replace(" ", "_") for item in specific_clothing]
    age, age_conf, age_evidence = infer_age_gender_aware(
        tag_probs, gender, body_type, breast_size, clothing_item_names
    )
    output["age"] = age
    output["age_confidence"] = round(age_conf, 3)
    output["age_evidence"] = age_evidence

    # ---- Expression ----
    # OPTIMIZATION #8: Use pre-filtered high_conf_tags
    expression_candidates = {
        t: high_conf_tags[t] for t in EXPRESSION_TAGS if t in high_conf_tags
    }

    if expression_candidates:
        sorted_expressions = sorted(expression_candidates.items(), key=lambda x: x[1], reverse=True)
        output["expressions"] = [
            {"expression": expr, "confidence": round(conf, 3)}
            for expr, conf in sorted_expressions[:MAX_EXPRESSIONS]
        ]
    else:
        output["expressions"] = [{"expression": "unknown", "confidence": 1.0}]

    # ---- Accessories ----
    accessory_tags = {t for t in tag_probs if t in ACCESSORY_TAGS}
    accessories = _select_top_n(tag_probs, accessory_tags, MAX_ACCESSORIES)
    output["accessories"] = accessories

    # OPTIMIZATION #6: Removed cultural_elements and ethnicity_estimate

    # ---- Anime Origin ----
    anime_origin, ao_conf = _determine_anime_origin(tag_probs)
    output["anime_origin"] = anime_origin
    output["anime_origin_confidence"] = round(ao_conf, 3)

    return output


print("✓ Projector code loaded")

✓ Projector code loaded


In [6]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from typing import Dict, List


class ImprovedCLIPTagger:
    """
    Optimized zero-shot CLIP tagger with cached text embeddings
    """

    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = None
        self.processor = None
        self.is_loaded = False

        # Text embeddings cache (computed once during load_model)
        self.text_embeddings_cache = {}  # {category: (tags_list, embeddings_tensor)}

        # Attribute templates (same as before)
        self.attribute_templates = {
            "gender": {
                "1girl": "a photo of a single female anime character",
                "1boy": "a photo of a single male anime character"
            },
            "age": {
                "child": "a photo of a child anime character",
                "teen": "a photo of a teenage anime character",
                "young_adult": "a photo of a young adult anime character",
                "middle_aged": "a photo of a middle-aged anime character",
                "elderly": "a photo of an elderly anime character"
            },
            "hair_color": {
                "black_hair": "anime character with black hair",
                "brown_hair": "anime character with brown hair",
                "blonde_hair": "anime character with blonde hair",
                "white_hair": "anime character with white hair",
                "red_hair": "anime character with red hair",
                "pink_hair": "anime character with pink hair",
                "blue_hair": "anime character with blue hair",
                "green_hair": "anime character with green hair",
                "purple_hair": "anime character with purple hair",
                "silver_hair": "anime character with silver hair",
                "orange_hair": "anime character with orange hair",
                "grey_hair": "anime character with grey hair"
            },
            "hair_length": {
                "long_hair": "anime character with long hair",
                "short_hair": "anime character with short hair",
                "medium_hair": "anime character with medium length hair",
                "very_long_hair": "anime character with very long hair"
            },
            "hair_style": {
                "ponytail": "anime character with ponytail hairstyle",
                "twintails": "anime character with twin tails hairstyle",
                "braid": "anime character with braided hair",
                "bun": "anime character with hair bun",
                "straight_hair": "anime character with straight hair",
                "wavy_hair": "anime character with wavy hair",
                "curly_hair": "anime character with curly hair",
                "ahoge": "anime character with ahoge (hair antenna)"
            },
            "eye_color": {
                "blue_eyes": "anime character with blue eyes",
                "brown_eyes": "anime character with brown eyes",
                "green_eyes": "anime character with green eyes",
                "red_eyes": "anime character with red eyes",
                "purple_eyes": "anime character with purple eyes",
                "yellow_eyes": "anime character with yellow eyes",
                "pink_eyes": "anime character with pink eyes",
                "orange_eyes": "anime character with orange eyes",
                "grey_eyes": "anime character with grey eyes"
            },
            "skin_tone": {
                "pale_skin": "anime character with very pale skin",
                "light_skin": "anime character with light skin",
                "tan_skin": "anime character with tan skin",
                "dark_skin": "anime character with dark skin",
                "brown_skin": "anime character with brown skin"
            },
            "body_type": {
                "slim": "slim anime character",
                "athletic": "athletic anime character",
                "muscular": "muscular anime character",
                "curvy": "curvy anime character",
                "petite": "petite anime character",
                "chubby": "chubby anime character",
                "tall": "tall anime character",
                "short": "short anime character"
            },
            "expression": {
                "smile": "smiling anime character",
                "angry": "angry anime character",
                "sad": "sad anime character",
                "surprised": "surprised anime character",
                "neutral": "anime character with neutral expression",
                "happy": "happy anime character",
                "serious": "serious anime character",
                "blush": "blushing anime character",
                "wink": "winking anime character",
                "crying": "crying anime character"
            },
            "clothing": {
                "school_uniform": "anime character wearing school uniform",
                "casual": "anime character in casual clothes",
                "dress": "anime character wearing a dress",
                "suit": "anime character wearing a suit",
                "kimono": "anime character wearing kimono",
                "swimsuit": "anime character in swimsuit",
                "uniform": "anime character in uniform",
                "hoodie": "anime character wearing hoodie",
                "coat": "anime character wearing coat"
            },
            "accessories": {
                "glasses": "anime character wearing glasses",
                "hat": "anime character wearing a hat",
                "ribbon": "anime character with ribbon",
                "bow": "anime character with bow",
                "necklace": "anime character wearing necklace",
                "earrings": "anime character wearing earrings",
                "headband": "anime character wearing headband",
                "scarf": "anime character wearing scarf",
                "gloves": "anime character wearing gloves",
                "backpack": "anime character with backpack"
            }
        }

        self.multi_value_categories = {"hair_style", "expression", "accessories"}

        print(f"✓ CLIP tagger initialized (lazy loading enabled)")

    def load_model(self):
        """Load CLIP model and PRE-COMPUTE all text embeddings"""
        if not self.is_loaded:
            print("Loading CLIP model...")
            self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(self.device)
            self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

            print("Pre-computing text embeddings (this happens ONCE)...")

            # Compute text embeddings for ALL categories at once
            with torch.no_grad():
                for category, tag_prompt_map in self.attribute_templates.items():
                    tags = list(tag_prompt_map.keys())
                    prompts = list(tag_prompt_map.values())

                    # Encode all prompts for this category
                    text_inputs = self.processor(
                        text=prompts,
                        return_tensors="pt",
                        padding=True
                    ).to(self.device)

                    text_features = self.model.get_text_features(**text_inputs)
                    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

                    # Cache: store (tag list, normalized embeddings)
                    self.text_embeddings_cache[category] = (tags, text_features)

                    print(f"  ✓ Cached {len(tags)} embeddings for '{category}'")

            self.is_loaded = True
            print(f"✓ CLIP model ready on {self.device}")

    def predict(self, image, threshold=0.15):
        """
        OPTIMIZED: Only encodes image once, uses cached text embeddings
        """
        self.load_model()

        # 1. Encode image (ONLY forward pass needed now!)
        inputs = self.processor(
            images=image,
            return_tensors="pt",
            padding=True
        ).to(self.device)

        with torch.no_grad():
            image_features = self.model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # 2. Compare with cached text embeddings (NO text encoding!)
        tag_probs = {}

        for category in self.attribute_templates.keys():
            tags, text_features = self.text_embeddings_cache[category]

            # Compute similarity with pre-computed embeddings
            similarities = (image_features @ text_features.T).squeeze(0)
            probs = torch.softmax(similarities * 100, dim=0)

            # Apply threshold logic
            if category in self.multi_value_categories:
                for tag, prob in zip(tags, probs):
                    prob_val = float(prob)
                    if prob_val >= threshold:
                        tag_probs[tag] = prob_val
            else:
                max_idx = torch.argmax(probs).item()
                max_prob = float(probs[max_idx])
                if max_prob >= threshold:
                    tag_probs[tags[max_idx]] = max_prob

        return tag_probs

    def predict_batch(self, images, threshold=0.15):
        """
        Batch processing with cached embeddings
        """
        self.load_model()

        inputs = self.processor(
            images=images,
            return_tensors="pt",
            padding=True
        ).to(self.device)

        with torch.no_grad():
            image_features = self.model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        results = []

        for img_feat in image_features:
            tag_probs = {}

            for category in self.attribute_templates.keys():
                tags, text_features = self.text_embeddings_cache[category]

                similarities = (img_feat @ text_features.T)
                probs = torch.softmax(similarities * 100, dim=0)

                if category in self.multi_value_categories:
                    for tag, prob in zip(tags, probs):
                        prob_val = float(prob)
                        if prob_val >= threshold:
                            tag_probs[tag] = prob_val
                else:
                    max_idx = torch.argmax(probs).item()
                    max_prob = float(probs[max_idx])
                    if max_prob >= threshold:
                        tag_probs[tags[max_idx]] = max_prob

            results.append(tag_probs)

        return results

    def unload_model(self):
        """Free memory"""
        if self.is_loaded:
            del self.model
            del self.processor
            self.text_embeddings_cache.clear()
            torch.cuda.empty_cache()
            self.model = None
            self.processor = None
            self.is_loaded = False
            print("✓ CLIP model unloaded from memory")


In [7]:
# app.py

import gradio as gr
from gradio_client import Client, handle_file
from PIL import Image
import numpy as np
import json
import time
from typing import Dict, Any, Tuple, List, Optional
import tempfile
import os

# # Import your optimized projector module
# from optimized_paste import run_projector

# # Import improved CLIP tagger (lazy load)
# from clip_tagger import ImprovedCLIPTagger

# ------------------------------
# ------- Configuration --------
# ------------------------------

# Initialize DeepDanbooru client
DEEPDANBOORU_CLIENT = Client("hysts/DeepDanbooru")

# Initialize CLIP tagger (lazy load - model loads on first use)
CLIP_TAGGER = ImprovedCLIPTagger()

# ------------------------------
# ------- Utilities ------------
# ------------------------------

def run_deepdanbooru(image: Image, score_threshold: float = 0.1) -> Dict[str, float]:
    """
    Calls hysts/DeepDanbooru via HuggingFace Gradio API
    Returns: dict {tag: confidence}
    """
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        image.save(tmp.name)
        tmp_path = tmp.name

    try:
        result = DEEPDANBOORU_CLIENT.predict(
            image=handle_file(tmp_path),
            score_threshold=score_threshold,
            api_name="/predict"
        )

        tag_probs = {
            item["label"]: float(item["confidence"])
            for item in result[0]["confidences"]
            if item["label"] is not None
        }

        return tag_probs

    finally:
        os.remove(tmp_path)


def run_clip_tagger(image: Image, threshold: float = 0.15) -> Dict[str, float]:
    """
    Calls improved CLIP tagger
    Returns: dict {tag: confidence}
    """
    return CLIP_TAGGER.predict(image, threshold=threshold)


def format_simple_output(structured: Dict[str, Any]) -> Dict[str, str]:
    """
    Convert the structured output to the simplified format
    """
    output = {}

    age = structured.get("age", "unknown")
    if age != "unknown":
        output["Age"] = " ".join(word.capitalize() for word in age.split("_"))
    else:
        output["Age"] = "Unknown"

    gender = structured.get("gender", "unknown")
    output["Gender"] = gender.capitalize() if gender != "unknown" else "Unknown"

    hair_styles = structured.get("hair_styles", [])
    if hair_styles and len(hair_styles) > 0:
        style = hair_styles[0]["value"]
        output["Hair Style"] = " ".join(word.capitalize() for word in style.split("_"))
    else:
        output["Hair Style"] = "None"

    hair_color = structured.get("hair_color", "unknown")
    output["Hair Color"] = hair_color.capitalize() if hair_color != "unknown" else "Unknown"

    hair_length = structured.get("hair_length", "unknown")
    output["Hair Length"] = hair_length.capitalize() if hair_length != "unknown" else "Unknown"

    eye_color = structured.get("eye_color", "unknown")
    output["Eye Color"] = eye_color.capitalize() if eye_color != "unknown" else "Unknown"

    skin_tone = structured.get("skin_tone", "unknown")
    output["Skin Tone"] = skin_tone.replace("_", " ").title() if skin_tone != "unknown" else "Unknown"

    body_type = structured.get("body_type", "unknown")
    output["Body Type"] = body_type.capitalize() if body_type != "unknown" else "Unknown"

    clothing = structured.get("clothing", [])
    if clothing and len(clothing) > 0:
        dress_category = clothing[0]["category"]
        output["Dress"] = " ".join(word.capitalize() for word in dress_category.split("_"))
    else:
        clothing_items = structured.get("clothing_items", [])
        if clothing_items and len(clothing_items) > 0:
            output["Dress"] = clothing_items[0]["item"].title()
        else:
            output["Dress"] = "Unknown"

    expressions = structured.get("expressions", [])
    if expressions and len(expressions) > 0:
        expr = expressions[0]["expression"]
        output["Expression"] = expr.replace("_", " ").title() if expr != "unknown" else "Unknown"
    else:
        output["Expression"] = "Unknown"

    return output


def flatten_attributes(structured: Dict[str, Any]) -> List[List[str]]:
    """
    Flatten structured attributes into a table for DataFrame view
    Returns 2D list for Gradio Dataframe
    """
    rows = []

    if structured.get("image_status") == "ambiguous_multi_character":
        rows.append(["Image status", "Multiple characters detected", "0.000"])
        rows.append(["Character count", str(structured.get("character_count", "unknown")), "0.000"])
        return rows

    if structured.get("image_status") == "no_character_detected":
        rows.append(["Image status", "No character detected", "0.000"])
        rows.append(["Scene type", structured.get("scene_type", "unknown"), "0.000"])
        return rows

    rows.append(["Gender", structured.get("gender", "unknown"), f"{structured.get('gender_confidence', 0.0):.3f}"])
    rows.append(["Age", structured.get("age", "unknown"), f"{structured.get('age_confidence', 0.0):.3f}"])
    rows.append(["Eye color", structured.get("eye_color", "unknown"), f"{structured.get('eye_color_confidence', 0.0):.3f}"])
    rows.append(["Hair color", structured.get("hair_color", "unknown"), f"{structured.get('hair_color_confidence', 0.0):.3f}"])
    rows.append(["Hair length", structured.get("hair_length", "unknown"), f"{structured.get('hair_length_confidence', 0.0):.3f}"])

    for i, style in enumerate(structured.get("hair_styles", []), start=1):
        rows.append([f"Hair style {i}", style.get("value", "unknown"), f"{style.get('confidence', 0.0):.3f}"])

    rows.append(["Skin tone", structured.get("skin_tone", "unknown"), f"{structured.get('skin_tone_confidence', 0.0):.3f}"])
    rows.append(["Body type", structured.get("body_type", "unknown"), f"{structured.get('body_type_confidence', 0.0):.3f}"])

    for i, item in enumerate(structured.get("clothing", []), start=1):
        rows.append([f"Clothing category {i}", item.get("category", "unknown"), f"{item.get('confidence', 0.0):.3f}"])

    for i, item in enumerate(structured.get("clothing_items", []), start=1):
        rows.append([f"Clothing item {i}", item.get("item", "unknown"), f"{item.get('confidence', 0.0):.3f}"])

    for i, expr in enumerate(structured.get("expressions", []), start=1):
        rows.append([f"Expression {i}", expr.get("expression", "unknown"), f"{expr.get('confidence', 0.0):.3f}"])

    for i, acc in enumerate(structured.get("accessories", []), start=1):
        rows.append([f"Accessory {i}", acc.get("value", "unknown"), f"{acc.get('confidence', 0.0):.3f}"])

    rows.append(["Anime origin", structured.get("anime_origin", "unknown"), f"{structured.get('anime_origin_confidence', 0.0):.3f}"])

    return rows


def tags_to_sorted_list(tag_probs: Dict[str, float], top_k: int = 50) -> List[List[str]]:
    """
    Convert tag probabilities to sorted 2D list for Gradio Dataframe
    """
    items = sorted(tag_probs.items(), key=lambda x: x[1], reverse=True)
    return [[t[0], f"{t[1]:.4f}"] for t in items[:top_k]]


# ------------------------------
# ------- Inference API -------
# ------------------------------

def analyze_image_inference(
    image: Image,
    tagger_choice: str,
    show_raw_tags: bool
) -> Tuple:
    """
    Main orchestrator for multi-tagger inference
    Returns: Multiple outputs based on tagger choice
    """
    if image is None:
        empty_table = [["No data", "", ""]]
        empty_dict = {}
        status = "⚠️ Please upload an image first"

        # Return empty results for all possible outputs
        return (
            empty_dict, empty_dict, empty_table, empty_table,  # DeepDanbooru
            empty_dict, empty_dict, empty_table, empty_table,  # CLIP
            status
        )

    start = time.time()

    try:
        # Initialize results
        dd_simple, dd_full, dd_table, dd_tags = {}, {}, [["No data", "", ""]], [["N/A", ""]]
        clip_simple, clip_full, clip_table, clip_tags = {}, {}, [["No data", "", ""]], [["N/A", ""]]

        # Run DeepDanbooru
        if tagger_choice in ["DeepDanbooru", "Both"]:
            tag_probs_dd = run_deepdanbooru(image)
            structured_dd = run_projector(tag_probs_dd)
            dd_simple = format_simple_output(structured_dd)
            dd_full = structured_dd
            dd_table = flatten_attributes(structured_dd)
            dd_tags = tags_to_sorted_list(tag_probs_dd, top_k=50) if show_raw_tags else [["Enable 'Show Raw Tags' to view", ""]]

        # Run CLIP
        if tagger_choice in ["CLIP", "Both"]:
            tag_probs_clip = run_clip_tagger(image)
            structured_clip = run_projector(tag_probs_clip)
            clip_simple = format_simple_output(structured_clip)
            clip_full = structured_clip
            clip_table = flatten_attributes(structured_clip)
            clip_tags = tags_to_sorted_list(tag_probs_clip, top_k=50) if show_raw_tags else [["Enable 'Show Raw Tags' to view", ""]]

        latency = time.time() - start

        # Add metadata
        if dd_full:
            dd_full["_meta"] = {
                "tagger": "DeepDanbooru",
                "inference_latency_s": round(latency, 3)
            }
        if clip_full:
            clip_full["_meta"] = {
                "tagger": "CLIP",
                "inference_latency_s": round(latency, 3)
            }

        status_msg = f"✅ Analysis complete in {latency:.2f}s using {tagger_choice}"

        return (
            dd_simple, dd_full, dd_table, dd_tags,
            clip_simple, clip_full, clip_table, clip_tags,
            status_msg
        )

    except Exception as e:
        error_msg = f"❌ Error during analysis: {str(e)}"
        empty_table = [["Error", str(e), ""]]
        empty_dict = {}

        return (
            empty_dict, empty_dict, empty_table, [["Error", ""]],
            empty_dict, empty_dict, empty_table, [["Error", ""]],
            error_msg
        )


# ------------------------------
# ------- Gradio UI ------------
# ------------------------------

def build_ui():
    custom_css = """
    .gradio-container {
        font-family: 'Inter', 'Segoe UI', Roboto, sans-serif;
    }

    .header-title {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
        font-size: 2.5em;
        font-weight: 700;
        margin-bottom: 0.5em;
    }

    button.primary {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
        border: none !important;
        font-weight: 600 !important;
        transition: transform 0.2s !important;
    }

    button.primary:hover {
        transform: translateY(-2px);
        box-shadow: 0 4px 12px rgba(102, 126, 234, 0.4) !important;
    }

    .input-section, .output-section {
        border-radius: 16px;
        padding: 20px;
        background: rgba(255, 255, 255, 0.03);
        backdrop-filter: blur(10px);
        border: 1px solid rgba(255, 255, 255, 0.1);
    }

    .status-message {
        padding: 12px;
        border-radius: 8px;
        font-weight: 500;
        text-align: center;
        margin-top: 10px;
    }
    """

    with gr.Blocks(css=custom_css, theme=gr.themes.Default(primary_hue="purple", secondary_hue="blue")) as demo:

        gr.Markdown(
            """
            <div class="header-title">
            🎨 Anime Character Attribute Extractor
            </div>
            """,
            elem_classes="header-title"
        )

        gr.Markdown(
            """
            Upload an anime character image to extract detailed attributes using AI taggers.
            **Features**: Multi-tagger support • Gender-aware detection • Zero-shot CLIP inference • Confidence scoring
            """,
            elem_classes="subtitle"
        )

        # Main two-column layout
        with gr.Row():
            # LEFT COLUMN - Input
            with gr.Column(scale=1, elem_classes="input-section"):
                gr.Markdown("### 📥 Input")
                image_in = gr.Image(
                    type="pil",
                    label="Upload Anime Character Image",
                    height=400
                )

                # Tagger selection
                gr.Markdown("**Select Tagger**")
                tagger_radio = gr.Radio(
                    choices=["DeepDanbooru", "CLIP", "Both"],
                    value="DeepDanbooru",
                    label="Choose inference model",
                    info="DeepDanbooru: trained tagger | CLIP: zero-shot | Both: compare results"
                )

                analyze_btn = gr.Button(
                    "🔍 Analyze Image",
                    variant="primary",
                    size="lg"
                )

                with gr.Row():
                    cb_raw_tags = gr.Checkbox(
                        label="Show Raw Tags",
                        value=False,
                        elem_classes="checkbox-label"
                    )

                status_box = gr.Markdown(
                    "💡 Upload an image and click Analyze",
                    elem_classes="status-message"
                )

            # RIGHT COLUMN - Output
            with gr.Column(scale=1, elem_classes="output-section"):
                gr.Markdown("### 📊 Output")

                with gr.Tabs() as main_tabs:
                    # DeepDanbooru Results
                    with gr.Tab("DeepDanbooru Results"):
                        with gr.Tabs():
                            with gr.Tab("Simple Output"):
                                dd_simple_json = gr.JSON(label="Character Attributes (Simplified)")

                            with gr.Tab("Extensive JSON"):
                                dd_full_json = gr.JSON(label="Complete Structured Output")

                            with gr.Tab("Detailed Table"):
                                dd_table_output = gr.Dataframe(
                                    headers=["Attribute", "Value", "Confidence"],
                                    label="Attribute Breakdown",
                                    interactive=False,
                                    wrap=True
                                )

                    # CLIP Results
                    with gr.Tab("CLIP Results"):
                        with gr.Tabs():
                            with gr.Tab("Simple Output"):
                                clip_simple_json = gr.JSON(label="Character Attributes (Simplified)")

                            with gr.Tab("Extensive JSON"):
                                clip_full_json = gr.JSON(label="Complete Structured Output")

                            with gr.Tab("Detailed Table"):
                                clip_table_output = gr.Dataframe(
                                    headers=["Attribute", "Value", "Confidence"],
                                    label="Attribute Breakdown",
                                    interactive=False,
                                    wrap=True
                                )

        # Raw Tags Section
        with gr.Accordion("🏷️ Raw Tags (Evidence)", open=False):
            gr.Markdown("View the raw tags extracted by each tagger. Enable 'Show Raw Tags' above.")

            with gr.Tabs():
                with gr.Tab("DeepDanbooru Tags"):
                    dd_tags_output = gr.Dataframe(
                        headers=["Tag", "Probability"],
                        label="Top 50 DeepDanbooru Tags",
                        interactive=False,
                        wrap=True
                    )

                with gr.Tab("CLIP Tags"):
                    clip_tags_output = gr.Dataframe(
                        headers=["Tag", "Probability"],
                        label="Top 50 CLIP Tags",
                        interactive=False,
                        wrap=True
                    )

        # Footer
        gr.Markdown(
            """
            ---
            **ℹ️ About**:
            - **DeepDanbooru**: Trained on anime images, provides detailed tags
            - **CLIP**: Zero-shot vision-language model, no training needed
            - **Both**: Compare results side-by-side to see different perspectives

            **⚡ Performance**: CLIP uses lazy loading (only loads when selected). Optimized projector processes both outputs.
            """,
            elem_classes="footer"
        )

        # Event handler
        def on_analyze_click(image, tagger_choice, show_raw_tags):
            results = analyze_image_inference(image, tagger_choice, show_raw_tags)
            return results

        analyze_btn.click(
            fn=on_analyze_click,
            inputs=[image_in, tagger_radio, cb_raw_tags],
            outputs=[
                dd_simple_json, dd_full_json, dd_table_output, dd_tags_output,
                clip_simple_json, clip_full_json, clip_table_output, clip_tags_output,
                status_box
            ]
        )

    return demo


# Run the app
if __name__ == "__main__":
    demo = build_ui()
    demo.launch(share=True)


Loaded as API: https://hysts-deepdanbooru.hf.space ✔
✓ CLIP tagger initialized (lazy loading enabled)


/tmp/ipython-input-3839026506.py:314: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default(primary_hue="purple", secondary_hue="blue")) as demo:
/tmp/ipython-input-3839026506.py:314: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default(primary_hue="purple", secondary_hue="blue")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2365bd01ed6d1111db.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
